# Control: mismas variables en todos los modelos

En `02_modelo.ipynb`, XGBoost recibe `cp` y `alcaldia` como categorías, y la regresión lineal y el MLP no
(ellos usan estación, día de la semana y hora en one-hot, más las numéricas escaladas). `cp` y `alcaldia` son
atributos **fijos de cada estación**, así que en teoría la estación ya contiene esa información.

Este notebook lo comprueba: entrena XGBoost **con** y **sin** `cp` y `alcaldia`, con los mismos hiperparámetros,
la misma partición por fechas y el mismo periodo de prueba que el 02 (28 jul – 31 ago de 2026).

**Qué se reporta (decidido antes de correrlo):** las dos filas, sin elegir la mejor. La tabla principal del proyecto
usa las mismas variables en los tres modelos. Los hiperparámetros se buscaron con las variables completas, así que la
versión sin `cp` ni `alcaldia` queda ligeramente en desventaja.

**Cómo correrlo:** Restart & Run All. Necesita `trips_clean.parquet` y `data/processed/busqueda_hiperparametros.csv`.

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

import modelo as M

print("Periodo de los datos:", M.FECHA_INICIO, "a", M.FECHA_FIN)

Periodo de los datos: 2026-03-01 a 2026-08-31


In [2]:
validos = M.cargar_validos()
estaciones = M.cargar_estaciones()
tabla = M.construir_tabla(validos, estaciones)
train, test, corte = M.split_por_fechas(tabla)
print(f"Filas: {len(tabla):,} | train: {len(train):,} | prueba: {len(test):,} | corte: {corte.date()}")

h = pd.read_csv("../data/processed/busqueda_hiperparametros.csv").iloc[0]
params = {k: (h[k] if k == "objective" else int(h[k]) if k in ("max_depth", "min_child_weight") else float(h[k]))
          for k in M.ESPACIO_BUSQUEDA}
print("Hiperparámetros (versión 1):", params)

Filas: 2,842,368 | train: 2,275,296 | prueba: 567,072 | corte: 2026-07-28
Hiperparámetros (versión 1): {'learning_rate': 0.1, 'max_depth': 8, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_weight': 20, 'objective': 'count:poisson'}


## XGBoost con y sin código postal y alcaldía

In [3]:
sin_geo = [f for f in M.FEATURES if f not in ("cp", "alcaldia")]
variantes = {
    "XGBoost (con cp y alcaldía)": M.FEATURES,
    "XGBoost (sin cp ni alcaldía)": sin_geo,
}

filas = []
for nombre, feats in variantes.items():
    n_arb = M.elegir_arboles(train, params, feats)
    modelo = M.entrenar_xgb(train, params, n_arb, feats)
    p = np.clip(modelo.predict(test[feats]), 0, None)
    m = M.metricas(test[M.TARGET], p)
    filas.append({"modelo": nombre, "variables": len(feats), "arboles": n_arb, **m})
    print(f"{nombre}: {len(feats)} variables, {n_arb} árboles | RMSE {m['RMSE']:.3f} | MAE {m['MAE']:.3f} | sesgo {m['Sesgo']:+.3f}")

res = pd.DataFrame(filas).round(4)
res.to_csv("../data/processed/resultados_mismas_variables.csv", index=False)
res

XGBoost (con cp y alcaldía): 12 variables, 374 árboles | RMSE 2.370 | MAE 1.383 | sesgo +0.060
XGBoost (sin cp ni alcaldía): 10 variables, 340 árboles | RMSE 2.370 | MAE 1.382 | sesgo +0.054


,modelo,variables,arboles,RMSE,MAE,Sesgo
0,XGBoost (con cp y alcaldía),12,374,2.3696,1.3827,0.0597
1,XGBoost (sin cp ni alcaldía),10,340,2.3698,1.3823,0.0543


## Contra la tabla principal

In [4]:
principal = pd.read_csv("../data/processed/resultados_modelo.csv", index_col=0)
sin = res.set_index("modelo").loc["XGBoost (sin cp ni alcaldía)"]
con = res.set_index("modelo").loc["XGBoost (con cp y alcaldía)"]
print(f"RMSE con cp y alcaldía (aquí): {con['RMSE']:.4f} | en resultados_modelo.csv: {principal.loc['XGBoost', 'RMSE']:.4f}")
print(f"RMSE sin cp ni alcaldía:       {sin['RMSE']:.4f} | diferencia: {sin['RMSE'] - con['RMSE']:+.4f}")
print()
tabla_final = principal[["RMSE", "MAE", "Sesgo"]].copy()
tabla_final.loc["XGBoost"] = sin[["RMSE", "MAE", "Sesgo"]].to_numpy()
tabla_final.sort_values("RMSE").round(3)

RMSE con cp y alcaldía (aquí): 2.3696 | en resultados_modelo.csv: 2.3696
RMSE sin cp ni alcaldía:       2.3698 | diferencia: +0.0002



,RMSE,MAE,Sesgo
XGBoost,2.370,1.382,0.054
MLP,2.419,1.396,0.040
Regresión lineal,2.453,1.466,0.043
Media 4 semanas,2.494,1.449,-0.028
Promedio histórico,2.559,1.448,0.171
Semana pasada,3.055,1.747,-0.019


## Lectura de resultados

- **`cp` y `alcaldia` no aportan información.** Sin ellas, XGBoost da RMSE 2.3698 contra 2.3696 con ellas (diferencia de +0.0002); el MAE (1.3823 contra 1.3827) y el sesgo (+0.054 contra +0.060) tampoco cambian de forma relevante. Es lo esperado: ambas son atributos fijos de cada estación, y la estación ya entra como variable.
- **La comparación entre modelos es justa.** Con las mismas variables que la regresión lineal y el MLP (10 en vez de 12), XGBoost sigue siendo el mejor modelo: RMSE 2.370 contra 2.419 (MLP) y 2.453 (regresión lineal).
- **El control es consistente con el 02:** la fila "con cp y alcaldía" reproduce exactamente el RMSE de `resultados_modelo.csv` (2.3696).
- **Limitación.** Los hiperparámetros se buscaron con las 12 variables; la versión de 10 usa esos mismos (y elige 340 árboles en lugar de 374), así que no es una búsqueda nueva. Dado que la diferencia es de 0.0002, no se espera que cambie la conclusión.